# SadTalker Quick Demo (Modern Colab)

This notebook is updated for modern Colab runtimes, current CUDA-enabled PyTorch, and current pip resolver behavior.


## 1) Clone repository

In [ ]:

!git clone -b python312-modernization https://github.com/Inialpha/SadTalker.git
%cd SadTalker

# 2) Install dependencies

In [ ]:
!pip install -U pip setuptools wheel

!pip install torch torchvision torchaudio
!pip install opencv-python pillow scipy numpy imageio imageio-ffmpeg pydub tqdm safetensors

!pip install facexlib
!pip install gfpgan --no-deps
!pip install basicsr-fixed
!pip install -r requirements.txt

In [ ]:

%%bash

apt-get update -qq
apt-get install -y ffmpeg

## 3) Mount drive

## 4) Download model checkpoints

In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path("/content/SadTalker")
print("Downloading checkpoints with scripts/download_models.sh ...")
subprocess.run(["bash", str(REPO_DIR / "scripts" / "download_models.sh")], check=True, cwd=REPO_DIR)
print("Download complete")

## 5) Verify checkpoints

In [ ]:
from pathlib import Path

REPO_DIR = Path("/content/SadTalker")
ckpt = REPO_DIR / "checkpoints"

required = [
    "SadTalker_V0.0.2_256.safetensors",
    "SadTalker_V0.0.2_512.safetensors",
    "mapping_00109-model.pth.tar",
    "mapping_00229-model.pth.tar",
]

missing = [name for name in required if not (ckpt / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing checkpoints: {missing}")

print("Checkpoint verification passed")
for name in required:
    path = ckpt / name
    print(f"- {name}: {path.stat().st_size / (1024**2):.1f} MB")

## 5) Import libraries and verify runtime


In [ ]:
%xmode verbose

## 6) Load models


In [ ]:
import sys
from pathlib import Path

REPO_DIR = Path("/content/SadTalker")
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.utils.init_path import init_path
from src.utils.preprocess import CropAndExtract
from src.test_audio2coeff import Audio2Coeff
from src.facerender.animate import AnimateFromCoeff

checkpoint_dir = REPO_DIR / "checkpoints"
config_dir = REPO_DIR / "src" / "config"
device = "cuda" if __import__("torch").cuda.is_available() else "cpu"

paths = init_path(str(checkpoint_dir), str(config_dir), 256, False, "crop")
print("Resolved model paths:")
for k, v in paths.items():
    print(f"- {k}: {v}")

preprocess_model = CropAndExtract(paths, device)
audio_to_coeff = Audio2Coeff(paths, device)
animate_from_coeff = AnimateFromCoeff(paths, device)
print("Model components loaded on", device)

In [16]:
!git pull

Already up to date.


In [ ]:

from google.colab import files
from pathlib import Path
import shutil

INPUT_DIR = Path("examples/user_inputs")
INPUT_DIR.mkdir(parents=True, exist_ok=True)

print("📷 Upload source image")
img = files.upload()
img_name = next(iter(img))
source_image = INPUT_DIR / img_name
shutil.move(img_name, source_image)

print("🎤 Upload driving audio")
aud = files.upload()
aud_name = next(iter(aud))
driven_audio = INPUT_DIR / aud_name
shutil.move(aud_name, driven_audio)

print("Image:", source_image)
print("Audio:", driven_audio)

In [ ]:
!python inference.py \
    --driven_audio "{driven_audio}" \
    --source_image "{source_image}" \
    --result_dir results/quick_demo \
    --still \
    --preprocess full \
    --enhancer none

using safetensor as default
3DMM Extraction for source image
landmark Det:: 100% 1/1 [00:00<00:00,  6.33it/s]
3DMM Extraction In Video::   0% 0/1 [00:00<?, ?it/s]<class 'int'> 256
<class 'int'> 256
<class 'numpy.float64'> 1.0606139716921068
<class 'numpy.ndarray'> [127.83213254 120.69878876]
<class 'numpy.float64'> 127.83213253907414
<class 'numpy.float64'> 120.69878876186085
3DMM Extraction In Video:: 100% 1/1 [00:00<00:00,  9.11it/s]
mel:: 100% 4121/4121 [00:00<00:00, 42260.12it/s]
audio2exp:: 100% 413/413 [00:00<00:00, 471.65it/s]
Face Renderer::   3% 71/2061 [00:46<22:13,  1.49it/s]

In [26]:
from pathlib import Path
from IPython.display import Video, display
from google.colab import files

result_dir = Path("results/quick_demo")

video = max(result_dir.glob("*.mp4"), key=lambda p: p.stat().st_mtime)

display(Video(str(video), embed=True))

print(f"Generated video: {video}")

files.download(str(video))

Generated video: results/quick_demo/2026_08_02_13.08.24.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7) Run inference


## 8) Display results


In [ ]:
from pathlib import Path
from IPython.display import Video, display

REPO_DIR = Path("/content/SadTalker")
videos = sorted((REPO_DIR / "results" / "quick_demo").glob("*.mp4"), key=lambda p: p.stat().st_mtime)
if not videos:
    raise FileNotFoundError("No output video found in /content/SadTalker/results/quick_demo")

latest = videos[-1]
print("Latest output:", latest)
display(Video(str(latest), embed=True, width=512))